In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import netCDF4 as nc
from datetime import datetime
import statsmodels.api as sm
from shapely.geometry import LineString, Point

import sys
sys.path.insert(0,"../modules")
import fortsa


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/apps/python/3.10/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/apps/python/3.10/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/blue/olabarrieta/scottleeyoung/jupyter_venv/jvenv/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/blue/olabarrieta/scottleeyoung/jupyter_venv/jvenv/lib/python3.10/site-packages/traitlets/config/application.py", line 1075

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



ImportError: numpy.core.multiarray failed to import

In [ ]:
def interpolate_nans(x):
    """
    Linearly interpolate NaNs in a 1D NumPy array.

    Parameters:
    x : array-like
        1D array with potential NaNs.

    Returns:
    np.ndarray
        Array with NaNs linearly interpolated.
    """
    x = np.asarray(x, dtype=float)  # ensure array is float type for NaNs
    nans = np.isnan(x)
    if nans.any():
        not_nan = ~nans
        x[nans] = np.interp(
            np.flatnonzero(nans),           # indices of NaNs
            np.flatnonzero(not_nan),        # indices of non-NaNs
            x[not_nan]                      # non-NaN values
        )
    return x

def extract_datetime(timestamps):
    '''
    This function converts timestamps (date-time objects with data type datetime64['ns']) to date-time strings 
    with format '%Y-%m-%d %H:%M:%S'.
    '''
    N = timestamps.size
    timestamps = timestamps.astype(str)
    date_time = np.empty(N, dtype=object)
    for i in range(N):
        date_time[i] = timestamps[i][:10] + ' ' + timestamps[i][11:19]
    return date_time

def datetime_to_ordinal(dt):
    '''
    This function converts a datetime object to an ordinal object while considering the time of day.
    '''
    start_of_year = dt.replace(month=1, day=1, hour=0, minute=0, second=0, microsecond=0)
    days_elapsed = (dt - start_of_year).days
    seconds_elapsed = dt.hour * 3600. + dt.minute * 60. + dt.second + dt.microsecond / 1e6
    ordinal_date = start_of_year.toordinal() + days_elapsed + seconds_elapsed / 86400.
    return ordinal_date

def str_datetime_2_ord(str_dt, ref_dt, fmt):
    '''
    This function converts date-time strings to ordinals.
    '''
    N = str_dt.size
    ref_ord = datetime_to_ordinal(datetime.strptime(ref_dt, fmt) )
    ordinal = np.zeros(N, dtype=np.float64)
    for i in range(N):
        dt_obj = datetime.strptime(str_dt[i], fmt)
        ordinal[i] = datetime_to_ordinal(dt_obj) - ref_ord
    return ordinal

def interpolate_to_time(t0, t1, val):
    def linear_interp(x, x1, x2, y1, y2):
        """Simple linear interpolation or extrapolation"""
        if x2 == x1:
            return y1  # avoid division by zero
        return y1 + (x - x1) * (y2 - y1) / (x2 - x1)

    N = t0.size
    new_vals = np.full(N, np.nan)  # Initialize with NaNs
    t1_rounded = np.round(t1, 4)
    t0_rounded = np.round(t0, 4)

    for i in range(N):
        t_i = t0_rounded[i]

        # Exact match
        if t_i in t1_rounded:
            indx = np.where(t1_rounded == t_i)[0][0]
            new_vals[i] = val[indx]

        # Extrapolate left
        elif t0[i] < t1[0]:
            new_vals[i] = linear_interp(t0[i], t1[0], t1[1], val[0], val[1])

        # Extrapolate right
        elif t0[i] > t1[-1]:
            new_vals[i] = linear_interp(t0[i], t1[-2], t1[-1], val[-2], val[-1])

        # Interpolate
        else:
            try:
                indx1 = np.where(t1 >= t0[i])[0][0]
                indx2 = np.where(t1 <= t0[i])[0][-1]
                new_vals[i] = linear_interp(t0[i], t1[indx2], t1[indx1], val[indx2], val[indx1])
            except IndexError:
                new_vals[i] = np.nan  # fallback

        # reprint(f"{np.round(100 * (i+1) / N, 2)} %")

    return new_vals

def reg_trend(array, x):
    intercept = sm.add_constant(x)
    model = sm.OLS(array, intercept).fit()
    return model.fittedvalues

def get_indices_1d_coords(lon, lat, start, end):
    """
    Get indices of grid cells between two points along a straight line,
    ensuring only one index per column (longitude).

    Parameters:
        lon (1D array): Longitudes of the grid.
        lat (1D array): Latitudes of the grid.
        start (tuple): (lon, lat) of the starting point.
        end (tuple): (lon, lat) of the ending point.

    Returns:
        List of (i, j) index tuples where i = lat index, j = lon index.
    """

    # Make 2D grid
    lon_grid, lat_grid = np.meshgrid(lon, lat)
    
    # Create line between start and end
    line = LineString([start, end])

    selected_indices = {}

    # Loop over all grid cells
    for j, lon_val in enumerate(lon):
        for i, lat_val in enumerate(lat):
            point = Point(lon_val, lat_val)
            dist = line.distance(point)

            # Use a reasonable threshold to select points near the line
            threshold = max(np.diff(lon).mean(), np.diff(lat).mean()) * 0.75

            if dist < threshold:
                # Only keep the closest i for each j (1 index per column)
                if j not in selected_indices or dist < selected_indices[j][0]:
                    selected_indices[j] = (dist, (i, j))

    # Extract only the (i, j) parts, sorted by column (j)
    result = [val[1] for _, val in sorted(selected_indices.items())]
    return np.array(result)

def haversine(lat1, lon1, lat2, lon2):
    """
    Calculate the great-circle distance between two points on Earth using the Haversine formula.
    
    Parameters:
        lat1, lon1: Coordinates of the first point in decimal degrees.
        lat2, lon2: Coordinates of the second point in decimal degrees.
        
    Returns:
        Distance in kilometers.
    """
    # Radius of the Earth in kilometers
    R = 6378.0
    
    # Convert decimal degrees to radians
    lat1_rad = np.radians(lat1)
    lon1_rad = np.radians(lon1)
    lat2_rad = np.radians(lat2)
    lon2_rad = np.radians(lon2)
    
    # Differences in coordinates
    delta_lat = lat2_rad - lat1_rad
    delta_lon = lon2_rad - lon1_rad
    
    # Haversine formula
    a = np.sin(delta_lat / 2.0)**2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(delta_lon / 2.0)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    
    # Distance
    distance = R * c
    return distance

def periodogram(x, dt=1, oneside=True, win=False):
    '''
    This function calculates the Power Spectral Density (ESD / dt) via the periodogram method.
    source:     Thompson and Emery (2014) - Data Analysis Methods in Physical Oceanography
    x       - time series being analysed
    dt      - sampling interval
    '''

    N = len(x)
    bw = 1 / (N * dt)       # bandwidth

    # windowing time series if necessary using the Hamming window
    if win != False:
        win = np.hamming(N)
        x = x * win
    
    X = np.fft.fft(x) * dt
    norm = 1 / (N * dt)
    S = norm * np.abs(X) ** 2       # power spectral density (variance per unit frequency)
    f = np.fft.fftfreq(N, dt)       # frequency (cycles per unit time)
    var = S * bw        # variance of the time series

    # returning oneside (positive) if oneside = True
    if oneside == True:
        S = S[:N//2]
        f = f[:N//2]
        var = var[:N//2]
    
    return f, S, var

In [ ]:
estuary_res_file = "/orange/olabarrieta/scottleeyoung/slew/observations/SLE_obs.nc"
river_file = "/orange/olabarrieta/scottleeyoung/slew/Jan-01-2010_to_Jan-01-2021/forcings/SLEW_rivers-final_Jan-01-2010_to_Dec-31-2020.nc"
wind_speed_file = "/orange/olabarrieta/scottleeyoung/slew/observations/wind/SVWX_WSPE_Jan-01-2010_to_Jan-01-2021.csv"
wind_dir_file = "/orange/olabarrieta/scottleeyoung/slew/observations/wind/SVWX_WDIR_Jan-01-2010_to_Jan-01-2021.csv"
gs_tracer_file = "/orange/olabarrieta/scottleeyoung/slew/Jan-01-2010_to_Jan-01-2021/forcings/CopernicusMarine_data_Jan-01-2010_to_Jan-01-2021.nc"
air_temp_file = "/orange/olabarrieta/scottleeyoung/slew/observations/SVWX_airt_Jan-01-2010_to_Jan-01-2021.csv"
radt_file = "data/SVWX_RADT_Jan-01-2010_to_Jan-01-2021.csv"
cutoff_period = 21
kb_alpha = 4.8
dt_wind = 15/1440
dt_est = 15/1440
dt_tran = 1
dt_flow = 1
dt_air = 1
dt_radt = 1

# Filter Time Series

In [ ]:
# #====================================================
# River Discharge
# read data
ds = xr.open_dataset(river_file)
flow_time = ds['river_time'].values#[2557:2557+365]
flow = np.abs(ds['river_transport'].values)#[2557:2557+365,:])
ds.close()
total_flow = np.sum(flow,axis=1)
total_flow = interpolate_nans(total_flow)
flow_time = extract_datetime(flow_time)
flow_time = str_datetime_2_ord(flow_time, "1858-11-17 00:00:00", '%Y-%m-%d %H:%M:%S')#[2557:2557+365]

# interpolate data
N = int((flow_time[-1] - flow_time[0]) / dt_flow)
time_flow_interp = np.empty(N)
time_flow_interp[0] = flow_time[0]
for i in range(1, N):
    time_flow_interp[i] = time_flow_interp[i-1] + dt_flow
flow_interp = interpolate_to_time(time_flow_interp, flow_time, total_flow)

# filter data
flow_win = int(cutoff_period / np.diff(flow_time).mean())
if flow_win % 2 == 0:
    flow_win += 1
flow = fortsa.kaiser_bessel(flow_interp, flow_win, kb_alpha, "T")

In [ ]:
#====================================================
# Florida Current Transport
FC_tran = []
year = []
month = []
day = []
for i in range(11):
    FC_tran_file = f"/orange/olabarrieta/scottleeyoung/FC_transport_data/FC_cable_transport_{2010+i}_v2.dat"
    year.append(np.loadtxt(FC_tran_file, skiprows=42, usecols=0))
    month.append(np.loadtxt(FC_tran_file, skiprows=42, usecols=1))
    day.append(np.loadtxt(FC_tran_file, skiprows=42, usecols=2))
    FC_tran.append(np.loadtxt(FC_tran_file, skiprows=42, usecols=3))

FC_tran = np.concatenate(FC_tran)
year = np.concatenate(year)
month = np.concatenate(month)
day = np.concatenate(day)
FC_time = np.empty(year.size, dtype="<U19")
for i in range(year.size):
    FC_time[i] = f"{int(year[i])}-{int(month[i]):02d}-{int(day[i]):02d} 00:00:00"
FC_time = str_datetime_2_ord(FC_time, "1858-11-17 00:00:00", '%Y-%m-%d %H:%M:%S')

# interpolate data
FC_tran = interpolate_nans(FC_tran)
N = int((FC_time[-1] - FC_time[0]) / dt_tran)
time_tran_interp = np.empty(N)
time_tran_interp[0] = FC_time[0]
for i in range(1, N):
    time_tran_interp[i] = time_tran_interp[i-1] + dt_tran
tran_interp = interpolate_to_time(time_tran_interp, FC_time, FC_tran)

# filter data
FC_win = int(cutoff_period / dt_tran)
if FC_win % 2 == 0:
    FC_win += 1
tran_fil = fortsa.kaiser_bessel(tran_interp, FC_win, kb_alpha, "T")

In [ ]:
#====================================================
# Gulf Stream Tracers
# read data
ds = xr.open_dataset(gs_tracer_file, decode_timedelta=False)
temp_ds = ds['temp'].values
salt_ds = ds['salt'].values
lon_ds = ds['lon'].values
lat_ds = ds['lat'].values
depth = ds['depth'].values
gs_time = ds['time'].values[:-1]#/86400e9
ds.close()

# extract cross-section
start = (-78.819322,26.479712)
end = (-80.022494,26.684782)
indx = get_indices_1d_coords(lon_ds, lat_ds, start, end)

N = salt_ds.shape[0]
M = indx.shape[0]
temp = np.empty([N,depth.size,M])
salt = np.empty([N,depth.size,M])
lon = np.empty([M])
lat = np.empty([M])
for i in range(M):
    lon[i] = lon_ds[indx[i,1]]
    lat[i] = lat_ds[indx[i,0]]
    temp[:,:,i] = temp_ds[:,:,indx[i,0],indx[i,1]]
    salt[:,:,i] = salt_ds[:,:,indx[i,0],indx[i,1]]

# extract depth
temp[temp > 100] = np.nan
salt[salt > 100] = np.nan
B = haversine(lat[0], lon[0], lat[-1], lon[-1]) * 1000
h = np.empty(M)
indx = np.empty(M, dtype=int)
for i in range(M):
    indx[i] = np.where(np.isnan(np.nanmean(temp,axis=0)[:,i]))[0].min()
    h[i] = depth[indx[i]]
    
# depth average
dA = h.mean() * B / (indx.mean() * M)
A = h.mean() * B
temp = np.nansum(temp,axis=(1,2)) * dA / A
salt = np.nansum(salt,axis=(1,2)) * dA / A

gs_win = int(cutoff_period / dt_tran)
if gs_win % 2 == 0:
    gs_win += 1
temp_gs_fil = fortsa.kaiser_bessel(temp, gs_win, kb_alpha, "T")[:-1]
salt_gs_fil = fortsa.kaiser_bessel(salt, gs_win, kb_alpha, "T")[:-1]

/scratch/local/13361981/ipykernel_3109117/2051350248.py:37: RuntimeWarning: Mean of empty slice
  indx[i] = np.where(np.isnan(np.nanmean(temp,axis=0)[:,i]))[0].min()


In [ ]:
#====================================================
# Winds
# read data
svwx_wind_file = wind_speed_file
df = pd.read_csv(svwx_wind_file)
svwx = df.iloc[:,1].to_numpy()
df['Date-Time'] = pd.to_datetime(df['Date-Time'], format='%m-%d-%Y %H:%M', errors='coerce')
svwx_spe_time = df.iloc[:,0].to_numpy()
svwx_spe_time = extract_datetime(svwx_spe_time)
for i in range(svwx_spe_time.size):
    if svwx_spe_time[i] == 'NaT' or svwx_spe_time[i] == 'NaT ':
        svwx_spe_time[i] = np.nan
    else:
        svwx_spe_time[i] = str_datetime_2_ord(np.array([svwx_spe_time[i]]), "1858-11-17 00:00:00", '%Y-%m-%d %H:%M:%S')[0]
svwx_spe_time = np.array(svwx_spe_time, dtype=float)
svwx = interpolate_nans(svwx)

svwx_wind_dir_file = wind_dir_file
df = pd.read_csv(svwx_wind_dir_file)
svwx_dirr = df.iloc[:,1].to_numpy()[:-1]
df['Date-Time'] = pd.to_datetime(df['Date-Time'], format='%m-%d-%Y %H:%M', errors='coerce')
svwx_dir_time = df.iloc[:,0].to_numpy()
svwx_dir_time = extract_datetime(svwx_dir_time)
for i in range(svwx_dir_time.size):
    if svwx_dir_time[i] == 'NaT' or svwx_dir_time[i] == 'NaT ':
        svwx_dir_time[i] = np.nan
    else:
        svwx_dir_time[i] = str_datetime_2_ord(np.array([svwx_dir_time[i]]), "1858-11-17 00:00:00", '%Y-%m-%d %H:%M:%S')[0]
svwx_dir_time = np.array(svwx_dir_time, dtype=float)

svwx_time = np.empty(int(4018/dt_wind))
svwx_time[0] = svwx_spe_time[0]
for i in range(1, int(4018/dt_wind)):
    svwx_time[i] = svwx_time[i-1] + dt_wind
svwx_dirr = interpolate_nans(svwx_dirr)

svwx_spe = interpolate_to_time(svwx_time, svwx_spe_time, svwx)
svwx_dir = interpolate_to_time(svwx_time, svwx_dir_time, svwx_dirr)
svwx_u = -svwx_spe * np.sin(np.radians(svwx_dir))
svwx_v = -svwx_spe * np.cos(np.radians(svwx_dir))

# filter data
wind_win = int(cutoff_period / dt_wind)
if wind_win % 2 == 0:
    wind_win += 1
svwx_u_fil = fortsa.kaiser_bessel(svwx_u, wind_win, kb_alpha, "T")
svwx_v_fil = fortsa.kaiser_bessel(svwx_v, wind_win, kb_alpha, "T")

In [ ]:
#====================================================
# Air Temperature
# read data
df = pd.read_csv(air_temp_file, skiprows=33, na_values=' ')
airt = df.iloc[:,5].to_numpy()[:-1]
airt_time = df.iloc[_numpy()[:-1]
airt_time = str_datetime_2_ord(airt_time, "1858-11-17", "%Y-%m-%d")

# interpolate data
airt = interpolate_nans(airt)
N = int((airt_time[-1] - airt_time[0]) / dt_air)
time_air_interp = np.empty(N)
time_air_interp[0] = airt_time[0]
for i in range(1, N):
    time_air_interp[i] = time_air_interp[i-1] + dt_air
airt_interp = interpolate_to_time(time_air_interp, airt_time, airt)

# filter data
airt_win = int(cutoff_period / dt_air)
if airt_win % 2 == 0:
    airt_win += 1
airt_fil = fortsa.kaiser_bessel(airt_interp, airt_win, 4.8, "T")

In [ ]:
#====================================================
# Total Shortwave Radiation
# read data
df = pd.read_csv(radt_file,  na_values=["", " "])
radt_time = df.iloc[:,4].to_numpy()
radt = df.iloc[:,5].to_numpy()
radt_time = str_datetime_2_ord(radt_time, "17-Nov-1858", "%d-%b-%Y")

# interpolate data
radt = interpolate_nans(radt)
N = int((radt_time[-1] - radt_time[0]) / dt_radt)
time_radt_interp = np.empty(N)
time_radt_interp[0] = radt_time[0]
for i in range(1, N):
    time_radt_interp[i] = time_radt_interp[i-1] + dt_radt
radt_interp = interpolate_to_time(time_radt_interp, radt_time, radt)

# filter data
radt_win = int(cutoff_period / dt_radt)
if radt_win % 2 == 0:
    radt_win += 1
radt_fil = fortsa.kaiser_bessel(radt_interp, radt_win, 4.8, "T")

In [ ]:
#====================================================
# Write Driver Time Series to NetCDF
ds = nc.Dataset("data/filtered_driver_timeseries.nc", mode='w', format='NETCDF4')
ds.createDimension("airt_time", time_air_interp.size)
ds.createDimension("gs_time", gs_time.size)
ds.createDimension("gs_tran_time", time_tran_interp.size)
ds.createDimension("wind_time", svwx_time.size)
ds.createDimension("flow_time", time_flow_interp.size)
ds.createDimension("radt_time", time_radt_interp.size)

ds_flow_time = ds.createVariable("flow_time", "f8", ("flow_time",))
ds_airt_time = ds.createVariable("airt_time", "f8", ("airt_time",))
ds_gs_time = ds.createVariable("gs_time", "f8", ("gs_time",))
ds_gs_tran_time = ds.createVariable("gs_tran_time", "f8", ("gs_tran_time",))
ds_wind_time = ds.createVariable("wind_time", "f8", ("wind_time",))
ds_radt_time = ds.createVariable("radt_time", "f8", ("radt_time",))

ds_flow = ds.createVariable("flow", "f8", ("flow_time",))
ds_airt = ds.createVariable("airt", "f8", ("airt_time",))
ds_radt = ds.createVariable("radt", "f8", ("radt_time",))
ds_gs_salt = ds.createVariable("gs_salt", "f8", ("gs_time",))
ds_gs_temp = ds.createVariable("gs_temp", "f8", ("gs_time",))
ds_tran = ds.createVariable("gs_tran", "f8", ("gs_tran_time",))
ds_wind_u = ds.createVariable("wind_u", "f8", ("wind_time",))
ds_wind_v = ds.createVariable("wind_v", "f8", ("wind_time",))

ds_flow_time[:] = time_flow_interp
ds_airt_time[:] = time_air_interp
ds_radt_time[:] = time_radt_interp
ds_gs_time[:] = gs_time
ds_gs_tran_time[:] = time_tran_interp
ds_wind_time[:] = svwx_time
ds_radt[:] = radt_fil
ds_airt[:] = airt_fil
ds_gs_salt[:] = salt_gs_fil
ds_gs_temp[:] = temp_gs_fil
ds_tran[:] = tran_fil
ds_wind_u[:] = svwx_u_fil
ds_wind_v[:] = svwx_v_fil
ds_flow[:] = flow

ds.close()

In [ ]:
# # Extracting HR1 salt and temp from Jan-01-2015 onwards to avoid large data gap 2013--2014
# ds = xr.open_dataset(estuary_res_file, decode_timedelta=False)
# temp = ds['temp'].values
# salt = ds['salt'].values
# est_time = ds['time'].values
# ds.close()

# start = str_datetime_2_ord(np.array(["Jan-01-2015"]), "Nov-17-1858", "%b-%d-%Y")[0]
# indx = np.where(est_time <= start)[0][-1]
# hr1_temp = temp[indx:,2,:]
# hr1_salt = salt[indx:,2,:]
# hr1_time = est_time[indx:]

# ds = nc.Dataset("data/HR1_salt-temp.nc", mode='w', format='NETCDF4')
# ds.createDimension('time', hr1_time.size)
# ds.createDimension('two', 2)

# ds_time = ds.createVariable('time', 'f8', ('time',))
# ds_time.units = "days"

# ds_temp = ds.createVariable("temp", "f8", ("time", "two",))
# ds_salt = ds.createVariable("salt", "f8", ("time", "two",))

# ds_time[:] = hr1_time
# ds_temp[:] = hr1_temp
# ds_salt[:] = hr1_salt

# ds.close()

In [ ]:
#====================================================
# Estuary Response
# read data
ds = xr.open_dataset(estuary_res_file, decode_timedelta=False)
zeta = ds['zeta'].values#[::96,:][:-1,:]
temp = ds['temp'].values#[::96,:][:-1,:]
salt = ds['salt'].values#[::96,:][:-1,:]
est_time = ds['time'].values
ds.close()

ds = xr.open_dataset("data/HR1_salt-temp.nc", decode_times=False)
hr1_time = ds['time'].values
hr1_temp = ds['temp'].values
hr1_salt = ds['salt'].values
ds.close()

# filter data        
for i in range(2):
    zeta[:,i] = interpolate_nans(zeta[:,i])
for k in range(2):
    hr1_salt[:,k] = interpolate_nans(hr1_salt[:,k])
    hr1_temp[:,k] = interpolate_nans(hr1_temp[:,k])
    for i in range(3):
        temp[:,i,k] = interpolate_nans(temp[:,i,k])
        salt[:,i,k] = interpolate_nans(salt[:,i,k])

N = int((est_time[-1] - est_time[0]) / dt_est)
time_all = np.empty(N)
time_all[0] = est_time[0]
for i in range(1, N):
    time_all[i] = time_all[i-1] + dt_est

start = str_datetime_2_ord(np.array(["Jan-01-2015"]), "Nov-17-1858", "%b-%d-%Y")[0]
indx = np.where(np.round(time_all,4) <= start)[0][-1]
hr1_time_all = time_all[indx:]
N_hr1 = hr1_time_all.size

est_win = int(cutoff_period / np.diff(est_time).mean())
if est_win % 2 == 0:
    est_win += 1

zeta_interp = np.empty([N,2])
temp_interp = np.empty([N,3,2])
salt_interp = np.empty([N,3,2])
hr1_temp_interp = np.empty([N_hr1,2])
hr1_salt_interp = np.empty([N_hr1,2])
zeta_fil = np.empty_like(zeta_interp)
temp_fil = np.empty_like(temp_interp)
salt_fil = np.empty_like(salt_interp)
hr1_salt_fil = np.empty_like(hr1_salt_interp)
hr1_temp_fil = np.empty_like(hr1_temp_interp)
for k in range(2):
    zeta_interp[:,k] = interpolate_to_time(time_all, est_time, zeta[:,k])
    zeta_fil[:,k] = fortsa.kaiser_bessel(zeta_interp[:,k], est_win, kb_alpha, "T")

    hr1_salt_interp[:,k] = interpolate_to_time(hr1_time_all, hr1_time, hr1_salt[:,k])
    hr1_salt_fil[:,k] = fortsa.kaiser_bessel(hr1_salt_interp[:,k], est_win, kb_alpha, "T")
    hr1_temp_interp[:,k] = interpolate_to_time(hr1_time_all, hr1_time, hr1_temp[:,k])
    hr1_temp_fil[:,k] = fortsa.kaiser_bessel(hr1_temp_interp[:,k], est_win, kb_alpha, "T")

    for i in range(3):
        temp_interp[:,i,k] = interpolate_to_time(time_all, est_time, temp[:,i,k])
        salt_interp[:,i,k] = interpolate_to_time(time_all, est_time, salt[:,i,k])
        temp_fil[:,i,k] = fortsa.kaiser_bessel(temp_interp[:,i,k], est_win, kb_alpha, "T")
        salt_fil[:,i,k] = fortsa.kaiser_bessel(salt_interp[:,i,k], est_win, kb_alpha, "T")
        print(f"{k+1} of {2} | {i+1} of {3}")

ds = nc.Dataset("data/filtered_estuary_timeseries.nc", mode='w', format='NETCDF4')
ds.createDimension("ocean_time", time_all.size)
ds.createDimension("two", 2)
ds.createDimension("three", 3)

ds_time = ds.createVariable("ocean_time", "f8", ("ocean_time",))
ds_zeta = ds.createVariable("zeta", "f8", ("ocean_time", "two",))
ds_temp = ds.createVariable("temp", "f8", ("ocean_time", "three", "two",))
ds_salt = ds.createVariable("salt", "f8", ("ocean_time", "three", "two",))

ds_time[:] = time_all
ds_zeta[:] = zeta_fil
ds_temp[:] = temp_fil
ds_salt[:] = salt_fil

ds.close()

ds = nc.Dataset("data/HR1_salt-temp_filtered.nc", mode='w', format='NETCDF4')
ds.createDimension('time', hr1_time_all.size)
ds.createDimension('two', 2)

ds_time = ds.createVariable('time', 'f8', ('time',))
ds_time.units = "days"

ds_temp = ds.createVariable("temp", "f8", ("time", "two",))
ds_salt = ds.createVariable("salt", "f8", ("time", "two",))

ds_time[:] = hr1_time_all
ds_temp[:] = hr1_temp_fil
ds_salt[:] = hr1_salt_fil

ds.close()

1 of 2 | 1 of 3
1 of 2 | 2 of 3
1 of 2 | 3 of 3
2 of 2 | 1 of 3
2 of 2 | 2 of 3
2 of 2 | 3 of 3


# Detrend Time Series

In [ ]:
ds = xr.open_dataset("data/filtered_driver_timeseries.nc")
time_airt = ds["airt_time"].values
time_gs = ds["gs_time"].values
time_tran = ds["gs_tran_time"].values
time_wind = ds["wind_time"].values
time_flow = ds["flow_time"].values
time_radt = ds["radt_time"].values
radt = ds["radt"].values
airt = ds["airt"].values
flow = ds["flow"].values
gs_temp = ds["gs_temp"].values
gs_salt = ds["gs_salt"].values
gs_tran = ds["gs_tran"].values
wind_u = ds["wind_u"].values
wind_v = ds["wind_v"].values
ds.close()

ds = xr.open_dataset("data/filtered_estuary_timeseries.nc")
time_est = ds['ocean_time'].values
zeta = ds['zeta'].values
temp = ds['temp'].values
salt = ds['salt'].values
ds.close()

ds = xr.open_dataset("data/HR1_salt-temp_filtered.nc", decode_times=False)
hr1_time = ds['time'].values
hr1_temp = ds['temp'].values
hr1_salt = ds['salt'].values
ds.close()

# Interpolate
two_days = int(2 / dt_est)
gs_tran_interp = interpolate_to_time(time_est[:-two_days], time_tran, gs_tran)
gs_temp_interp = interpolate_to_time(time_est[:-two_days], time_gs, gs_temp)
gs_salt_interp = interpolate_to_time(time_est[:-two_days], time_gs, gs_salt)
flow_interp = interpolate_to_time(time_est[:-two_days], time_flow, flow)
airt_interp = interpolate_to_time(time_est[:-two_days], time_airt, airt)
radt_interp = interpolate_to_time(time_est[:-two_days], time_radt, radt)

# Trend
zeta_detrend = np.empty_like(zeta)
zeta_trend = np.empty_like(zeta)
hr1_salt_detrend = np.empty_like(hr1_salt)
hr1_salt_trend = np.empty_like(hr1_salt)
hr1_temp_detrend = np.empty_like(hr1_temp)
hr1_temp_trend = np.empty_like(hr1_temp)
for i in range(zeta.shape[1]):
    zeta_trend[:,i] = reg_trend(zeta[:,i], time_est)
    zeta_detrend[:,i] = zeta[:,i] - zeta_trend[:,i]

    hr1_salt_trend[:,i] = reg_trend(hr1_salt[:,i], hr1_time)
    hr1_salt_detrend[:,i] = hr1_salt[:,i] - hr1_salt_trend[:,i]
    hr1_temp_trend[:,i] = reg_trend(hr1_temp[:,i], hr1_time)
    hr1_temp_detrend[:,i] = hr1_temp[:,i] - hr1_temp_trend[:,i]

salt_detrend = np.empty_like(salt)
salt_trend = np.empty_like(salt)
temp_detrend = np.empty_like(temp)
temp_trend = np.empty_like(temp)
for i in range(salt.shape[1]):
    for j in range(salt.shape[2]):
        salt_trend[:,i,j] = reg_trend(salt[:,i,j], time_est)
        salt_detrend[:,i,j] = salt[:,i,j] - salt_trend[:,i,j]
        temp_trend[:,i,j] = reg_trend(temp[:,i,j], time_est)
        temp_detrend[:,i,j] = temp[:,i,j] - temp_trend[:,i,j]

gs_tran_trend = reg_trend(gs_tran_interp, np.arange(gs_tran_interp.size))
gs_temp_trend = reg_trend(gs_temp_interp, np.arange(gs_temp_interp.size))
gs_salt_trend = reg_trend(gs_salt_interp, np.arange(gs_salt_interp.size))
gs_tran_detrend = gs_tran_interp - gs_tran_trend
gs_temp_detrend = gs_temp_interp - gs_temp_trend
gs_salt_detrend = gs_salt_interp - gs_salt_trend

flow_trend = reg_trend(flow_interp, np.arange(flow_interp.size))
radt_trend = reg_trend(radt_interp, np.arange(radt_interp.size))
airt_trend = reg_trend(airt_interp, np.arange(airt_interp.size))
wind_u_trend = reg_trend(wind_u, np.arange(wind_u.size))
wind_v_trend = reg_trend(wind_v, np.arange(wind_v.size))
flow_detrend = flow_interp - flow_trend
radt_detrend = radt_interp - radt_trend
airt_detrend = airt_interp - airt_trend
wind_u_detrend = wind_u - wind_u_trend
wind_v_detrend = wind_v - wind_v_trend

In [ ]:
#====================================================
# Write Detrended Driver Time Series to NetCDF
ds = nc.Dataset("data/detrended_driver_timeseries.nc", mode='w', format='NETCDF4')
ds.createDimension("time", time_est[:-two_days].size)
ds_time = ds.createVariable("time", "f8", ("time",))

ds_flow = ds.createVariable("flow", "f8", ("time",))
ds_radt = ds.createVariable("radt", "f8", ("time",))
ds_airt = ds.createVariable("airt", "f8", ("time",))
ds_gs_salt = ds.createVariable("gs_salt", "f8", ("time",))
ds_gs_temp = ds.createVariable("gs_temp", "f8", ("time",))
ds_tran = ds.createVariable("gs_tran", "f8", ("time",))
ds_wind_u = ds.createVariable("wind_u", "f8", ("time",))
ds_wind_v = ds.createVariable("wind_v", "f8", ("time",))

ds_flow_trend = ds.createVariable("flow_trend", "f8", ("time",))
ds_radt_trend = ds.createVariable("radt_trend", "f8", ("time",))
ds_airt_trend = ds.createVariable("airt_trend", "f8", ("time",))
ds_gs_salt_trend = ds.createVariable("gs_salt_trend", "f8", ("time",))
ds_gs_temp_trend = ds.createVariable("gs_temp_trend", "f8", ("time",))
ds_tran_trend = ds.createVariable("gs_tran_trend", "f8", ("time",))
ds_wind_u_trend = ds.createVariable("wind_u_trend", "f8", ("time",))
ds_wind_v_trend = ds.createVariable("wind_v_trend", "f8", ("time",))

ds_time[:] = time_est[:-two_days]

ds_radt[:] = radt_detrend
ds_airt[:] = airt_detrend
ds_gs_salt[:] = gs_salt_detrend
ds_gs_temp[:] = gs_temp_detrend
ds_tran[:] = gs_tran_detrend
ds_wind_u[:] = wind_u_detrend[1:-two_days]
ds_wind_v[:] = wind_v_detrend[1:-two_days]
ds_flow[:] = flow_detrend

ds_radt_trend[:] = radt_trend
ds_airt_trend[:] = airt_trend
ds_gs_salt_trend[:] = gs_salt_trend
ds_gs_temp_trend[:] = gs_temp_trend
ds_tran_trend[:] = gs_tran_trend
ds_wind_u_trend[:] = wind_u_trend[1:-two_days]
ds_wind_v_trend[:] = wind_v_trend[1:-two_days]
ds_flow_trend[:] = flow_trend

ds.close()

In [ ]:
# Write Detrended Estuary Time Series to NetCDF File
ds = nc.Dataset("data/detrended_estuary_timeseries.nc", mode='w', format='NETCDF4')
ds.createDimension("time", time_est[:-two_days].size)
ds.createDimension("two", 2)
ds.createDimension("three", 3)

ds_time = ds.createVariable("time", "f8", ("time",))
ds_zeta = ds.createVariable("zeta", "f8", ("time", "two",))
ds_temp = ds.createVariable("temp", "f8", ("time", "three", "two",))
ds_salt = ds.createVariable("salt", "f8", ("time", "three", "two",))

ds_zeta_trend = ds.createVariable("zeta_trend", "f8", ("time", "two",))
ds_temp_trend = ds.createVariable("temp_trend", "f8", ("time", "three", "two",))
ds_salt_trend = ds.createVariable("salt_trend", "f8", ("time", "three", "two",))

ds_time[:] = time_est[:-two_days]
ds_zeta[:] = zeta_detrend[:-two_days,:]
ds_temp[:] = temp_detrend[:-two_days,:,:]
ds_salt[:] = salt_detrend[:-two_days,:,:]

ds_zeta_trend[:] = zeta_trend[:-two_days,:]
ds_temp_trend[:] = temp_trend[:-two_days,:,:]
ds_salt_trend[:] = salt_trend[:-two_days,:,:]

ds.close()

ds = nc.Dataset("data/HR1_salt-temp_detrended.nc", mode='w', format='NETCDF4')
ds.createDimension('time', hr1_time.size)
ds.createDimension('two', 2)

ds_time = ds.createVariable('time', 'f8', ('time',))
ds_time.units = "days"

ds_temp = ds.createVariable("temp", "f8", ("time", "two",))
ds_salt = ds.createVariable("salt", "f8", ("time", "two",))

ds_temp_trend = ds.createVariable("temp_trend", "f8", ("time", "two",))
ds_salt_trend = ds.createVariable("salt_trend", "f8", ("time", "two",))

ds_time[:] = hr1_time
ds_temp[:] = hr1_temp_detrend
ds_salt[:] = hr1_salt_detrend
ds_temp_trend[:] = hr1_temp_trend
ds_salt_trend[:] = hr1_salt_trend

ds.close()

# Fourier Analysis

In [ ]:
#====================================================
# Spectral Analysis of the Gulf Stream
f_tran, tran_psd, var_tran = periodogram(gs_tran_detrend, dt=dt_est) 
f_salt_gs, salt_gs_psd, var_salt_gs = periodogram(gs_salt_detrend, dt=dt_est) 
f_temp_gs, temp_gs_psd, var_temp_gs = periodogram(gs_temp_detrend, dt=dt_est) 
f_flow, flow_psd, var_flow = periodogram(flow_detrend, dt=dt_est) 
f_airt, airt_psd, var_airt = periodogram(airt_detrend, dt=dt_est) 
f_wind_u, wind_u_psd, var_wind_u = periodogram(wind_u_detrend[1:-two_days], dt=dt_est)
f_wind_v, wind_v_psd, var_wind_v = periodogram(wind_v_detrend[1:-two_days], dt=dt_est)

tran_psd = 0.5 * np.sqrt(2 * tran_psd * f_tran)
salt_gs_psd = 0.5 * np.sqrt(2 * salt_gs_psd * f_salt_gs)
temp_gs_psd = 0.5 * np.sqrt(2 * temp_gs_psd * f_temp_gs)
flow_psd = 0.5 * np.sqrt(2 * flow_psd * f_flow)
airt_psd = 0.5 * np.sqrt(2 * airt_psd * f_airt)
wind_u_psd = 0.5 * np.sqrt(2 * wind_u_psd * f_wind_u)
wind_v_psd = 0.5 * np.sqrt(2 * wind_v_psd * f_wind_v)

ds = nc.Dataset("data/driver_fourier.nc", mode='w', format='NETCDF4')
ds.createDimension("freq", f_tran.size)
ds_f = ds.createVariable("freq", "f8", ("freq",))
ds_flow = ds.createVariable("flow", "f8", ("freq",))
ds_airt = ds.createVariable("airt", "f8", ("freq",))
ds_gs_salt = ds.createVariable("gs_salt", "f8", ("freq",))
ds_gs_temp = ds.createVariable("gs_temp", "f8", ("freq",))
ds_tran = ds.createVariable("gs_tran", "f8", ("freq",))
ds_wind_u = ds.createVariable("wind_u", "f8", ("freq",))
ds_wind_v = ds.createVariable("wind_v", "f8", ("freq",))

ds_f[:] = f_tran
ds_airt[:] = airt_psd
ds_gs_salt[:] = salt_gs_psd
ds_gs_temp[:] = temp_gs_psd
ds_tran[:] = tran_psd
ds_wind_u[:] = wind_u_psd
ds_wind_v[:] = wind_v_psd
ds_flow[:] = flow_psd

ds.close()

In [ ]:
# spectral analysis for estuarine time series
N = time_est[:-two_days].size // 2
zeta_psd = np.empty([N,2])
zeta_var = np.empty([N,2])
f_zeta = np.empty([N,2])
for i in range(2):
    f_zeta[:,i], zeta_psd[:,i], zeta_var[:,i] = periodogram(zeta_detrend[:-two_days,i], dt_est)
    zeta_psd[:,i] = 0.5 * np.sqrt(2 * zeta_psd[:,i] * f_zeta[:,i])

temp_psd = np.empty([N,3,2])
temp_var = np.empty([N,3,2])
f_temp = np.empty([N,3,2])
salt_psd = np.empty([N,3,2])
salt_var = np.empty([N,3,2])
f_salt = np.empty([N,3,2])
for k in range(2):
    for i in range(3):
        f_temp[:,i,k], temp_psd[:,i,k], temp_var[:,i,k] = periodogram(temp[:-two_days,i,k], dt_est)
        f_salt[:,i,k], salt_psd[:,i,k], salt_var[:,i,k] = periodogram(salt[:-two_days,i,k], dt_est)
        temp_psd[:,i,k] = 0.5 * np.sqrt(2 * temp_psd[:,i,k] * f_temp[:,i,k])
        salt_psd[:,i,k] = 0.5 * np.sqrt(2 * salt_psd[:,i,k] * f_salt[:,i,k])

ds = nc.Dataset("data/estuary_fourier.nc", mode='w', format='NETCDF4')
ds.createDimension("freq", N)
ds.createDimension("two", 2)
ds.createDimension("three", 3)
ds_f = ds.createVariable("freq", "f8", ("freq",))
ds_zeta = ds.createVariable("zeta", "f8", ("freq","two",))
ds_salt = ds.createVariable("salt", "f8", ("freq","three","two",))
ds_temp = ds.createVariable("temp", "f8", ("freq","three","two",))
ds_f[:] = f_tran
ds_zeta[:] = zeta_psd
ds_salt[:] = salt_psd
ds_temp[:] = temp_psd
ds.close()